# Kadem-style GPR-EKF BMS Baseline for the Multi-HI Project

This Colab notebook implements a controlled **Kadem-style single-HI GPR-EKF BMS baseline** under the same Python/Colab environment used by the Multi-HI + MAW-GME experiments.

It is not the original MATLAB implementation. It is a controlled Python reimplementation that uses:

- normalized maximum IC peak (`pmax_norm`) as the single health indicator,
- a GPR transition model for online SoH prediction,
- a GPR measurement model from SoH to normalized Pmax,
- EKF prediction/update equations,
- leave-one-cell-out evaluation,
- optional intermittent measurement availability stress tests.

The notebook assumes that the three dataset notebooks have already produced `multi_hi_features.csv` files. It then runs the Kadem-style baseline on the same extracted feature tables and on the same Colab runtime.


In [ ]:
# ============================================================
# 0) GOOGLE DRIVE MOUNT
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped or unavailable:', e)


In [ ]:
# ============================================================
# 1) INSTALL / IMPORT DEPENDENCIES
# ============================================================
import os
import sys
import time
import math
import pickle
import warnings
import subprocess
import importlib.util
from pathlib import Path

warnings.filterwarnings('ignore')


def pip_install_if_missing(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('scipy', 'scipy'),
    ('scikit-learn', 'sklearn'),
    ('openpyxl', 'openpyxl'),
    ('tqdm', 'tqdm'),
]:
    pip_install_if_missing(pkg, imp)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from IPython.display import display


In [ ]:
# ============================================================
# 2) PATHS AND EXPERIMENT CONFIG
# ============================================================
DRIVE_ROOT = '/content/drive/MyDrive/Batarya'

DATASET_CONFIGS = {
    'Dataset 1': {
        'feature_csv': os.path.join(DRIVE_ROOT, 'Dataset_1', 'Dataset1_NMC_MAW_GME_results', 'features', 'multi_hi_features.csv'),
        # Kadem paper: Dataset 1 Pmax may be unavailable if 3.6-3.85 V is not covered.
        'pmax_window': 'narrow_36_385',
        'paper_gpr_ekf_rmse': 0.0188,
        'paper_gpr_ekf_mae': 0.0156,
    },
    'Dataset 2': {
        'feature_csv': os.path.join(DRIVE_ROOT, 'Oxford_Battery_Degradation_Dataset_2', 'features', 'multi_hi_features.csv'),
        # Kadem paper: Dataset 2 Pmax may be unavailable if 3.8-3.9 V is not covered.
        'pmax_window': 'very_narrow_38_39',
        'paper_gpr_ekf_rmse': 0.0045,
        'paper_gpr_ekf_mae': 0.0034,
    },
    'Dataset 3': {
        'feature_csv': os.path.join(DRIVE_ROOT, 'Dataset3_LFP_MAW_GME', 'features', 'multi_hi_features.csv'),
        # Kadem paper: Dataset 3 Pmax may be unavailable if 3.35-3.50 V is not covered.
        'pmax_window': 'paper_335_350',
        'paper_gpr_ekf_rmse': 0.0106,
        'paper_gpr_ekf_mae': 0.0084,
    },
}

RESULT_ROOT = os.path.join(DRIVE_ROOT, 'Kadem_GPR_EKF_BMS_Colab')
os.makedirs(RESULT_ROOT, exist_ok=True)

# Intermittent HI measurement availability probabilities.
# 1.0 is normal online replay. 0.8 matches the main intermittent test level in Kadem's paper.
AVAILABILITY_PROBS = [1.0, 0.8, 0.6, 0.4]
N_SEEDS_FOR_DROPOUT = 10
RANDOM_SEED = 42

# EKF numerical settings.
P0 = 0.01
DELTA_JAC = 1e-4
Q_FLOOR = 1e-7
R_FLOOR = 1e-7
CLIP_SOH = (0.0, 1.05)

print('RESULT_ROOT:', RESULT_ROOT)
for name, cfg in DATASET_CONFIGS.items():
    print(name, 'feature_csv exists:', os.path.exists(cfg['feature_csv']), '->', cfg['feature_csv'])


In [ ]:
# ============================================================
# 3) HELPERS
# ============================================================

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    return {
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'R2': float(r2_score(y_true, y_pred)) if len(np.unique(y_true)) > 1 else np.nan,
    }


def safe_cycle_column(df):
    candidates = ['cycle_number', 'cycle', 'rpt_index', 'RPT', 'rpt']
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f'No cycle/rpt column found. Available columns include: {list(df.columns)[:30]}')


def safe_cell_column(df):
    candidates = ['cell', 'battery', 'Battery', 'cell_id']
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError('No cell column found.')


def load_feature_table(dataset_name, cfg):
    path = cfg['feature_csv']
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Feature file not found for {dataset_name}: {path}"
            f"Run the corresponding Multi-HI/MAW-GME notebook first so that multi_hi_features.csv is created."
        )
    df = pd.read_csv(path)
    cell_col = safe_cell_column(df)
    cyc_col = safe_cycle_column(df)
    pmax_col = f"pmax_norm_{cfg['pmax_window']}"
    if pmax_col not in df.columns:
        # Fallback: show likely pmax columns.
        pcols = [c for c in df.columns if c.startswith('pmax_norm')]
        raise ValueError(f'{pmax_col} not found for {dataset_name}. Available pmax_norm columns: {pcols}')
    need = [cell_col, cyc_col, 'soh', pmax_col]
    out = df[need].copy()
    out = out.rename(columns={cell_col: 'cell', cyc_col: 'cycle_index', pmax_col: 'pmax_norm'})
    out['cell'] = out['cell'].astype(str)
    out['cycle_index'] = pd.to_numeric(out['cycle_index'], errors='coerce')
    out['soh'] = pd.to_numeric(out['soh'], errors='coerce')
    out['pmax_norm'] = pd.to_numeric(out['pmax_norm'], errors='coerce')
    out = out.dropna(subset=['cell', 'cycle_index', 'soh']).sort_values(['cell', 'cycle_index']).reset_index(drop=True)
    return out


def build_transition_training_data(train_df):
    X, y = [], []
    for cell, g in train_df.groupby('cell', sort=False):
        g = g.sort_values('cycle_index').dropna(subset=['soh', 'cycle_index']).reset_index(drop=True)
        if len(g) < 2:
            continue
        cycles = g['cycle_index'].values.astype(float)
        sohs = g['soh'].values.astype(float)
        for i in range(1, len(g)):
            u_prev = cycles[i-1]
            du = cycles[i] - cycles[i-1]
            if not np.isfinite(du) or du <= 0:
                # Some datasets may have repeated/irregular indices; use a small positive step.
                du = 1.0
            X.append([sohs[i-1], u_prev, du])
            y.append(sohs[i])
    return np.asarray(X, dtype=float), np.asarray(y, dtype=float)


def make_gpr(n_features, random_state=42):
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(
        length_scale=np.ones(n_features), length_scale_bounds=(1e-3, 1e3)
    ) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e1))
    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        alpha=1e-8,
        n_restarts_optimizer=2,
        random_state=random_state,
    )


class ScaledGPR:
    def __init__(self, n_features, random_state=42):
        self.scaler = StandardScaler()
        self.model = make_gpr(n_features, random_state=random_state)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.scaler.fit(X)
        Xs = self.scaler.transform(X)
        self.model.fit(Xs, y)
        return self

    def predict(self, X, return_std=False):
        X = np.asarray(X, dtype=float)
        Xs = self.scaler.transform(X)
        return self.model.predict(Xs, return_std=return_std)


def finite_jacobian_pred(pred_gpr, x, u_prev, du, delta=DELTA_JAC):
    xp = min(CLIP_SOH[1], x + delta)
    xm = max(CLIP_SOH[0], x - delta)
    yp = float(pred_gpr.predict([[xp, u_prev, du]])[0])
    ym = float(pred_gpr.predict([[xm, u_prev, du]])[0])
    denom = max(1e-12, xp - xm)
    return (yp - ym) / denom


def finite_jacobian_meas(meas_gpr, x, delta=DELTA_JAC):
    xp = min(CLIP_SOH[1], x + delta)
    xm = max(CLIP_SOH[0], x - delta)
    yp = float(meas_gpr.predict([[xp]])[0])
    ym = float(meas_gpr.predict([[xm]])[0])
    denom = max(1e-12, xp - xm)
    return (yp - ym) / denom


def pickle_size_kb(obj):
    try:
        return len(pickle.dumps(obj)) / 1024.0
    except Exception:
        return np.nan


In [ ]:
# ============================================================
# 4) KADEM-STYLE GPR-EKF ONLINE BMS BASELINE
# ============================================================

def fit_kadem_gpr_ekf_models(train_df, random_state=42):
    # Prediction/transition GPR: [previous SoH estimate, previous cycle/EFC proxy, delta cycle] -> current SoH
    Xp, yp = build_transition_training_data(train_df)
    if len(Xp) < 8:
        raise RuntimeError(f'Not enough transition samples for GPR prediction model: {len(Xp)}')
    pred_gpr = ScaledGPR(n_features=3, random_state=random_state).fit(Xp, yp)

    # Measurement GPR: SoH -> normalized Pmax.
    meas_df = train_df.dropna(subset=['soh', 'pmax_norm']).copy()
    if len(meas_df) < 8:
        raise RuntimeError(f'Not enough measurement samples for GPR measurement model: {len(meas_df)}')
    Xm = meas_df[['soh']].values.astype(float)
    ym = meas_df['pmax_norm'].values.astype(float)
    meas_gpr = ScaledGPR(n_features=1, random_state=random_state).fit(Xm, ym)
    return pred_gpr, meas_gpr


def run_kadem_gpr_ekf_on_cell(test_df, pred_gpr, meas_gpr, availability_prob=1.0, seed=42):
    rng = np.random.default_rng(seed)
    g = test_df.sort_values('cycle_index').reset_index(drop=True).copy()
    if len(g) == 0:
        return pd.DataFrame(), {}

    # Initial SoH is assumed to be known from beginning-of-life calibration.
    x = 1.0
    P = P0
    prev_cycle = float(g.loc[0, 'cycle_index'])

    rows = []
    update_times_ms = []

    for idx, row in g.iterrows():
        t0 = time.perf_counter()
        cycle = float(row['cycle_index'])
        y_true = float(row['soh'])
        z = row['pmax_norm'] if pd.notna(row['pmax_norm']) else np.nan

        if idx == 0:
            x_pred = x
            P_pred = P
            q = np.nan
            F = np.nan
        else:
            du = cycle - prev_cycle
            if not np.isfinite(du) or du <= 0:
                du = 1.0
            # Prediction step.
            pred_mean, pred_std = pred_gpr.predict([[x, prev_cycle, du]], return_std=True)
            x_pred = float(pred_mean[0])
            q = max(float(pred_std[0]) ** 2, Q_FLOOR)
            F = float(finite_jacobian_pred(pred_gpr, x, prev_cycle, du))
            P_pred = F * P * F + q
            x_pred = float(np.clip(x_pred, CLIP_SOH[0], CLIP_SOH[1]))

        measurement_available = bool(np.isfinite(z) and (rng.random() <= availability_prob))
        if measurement_available:
            h_mean, h_std = meas_gpr.predict([[x_pred]], return_std=True)
            h_pred = float(h_mean[0])
            r = max(float(h_std[0]) ** 2, R_FLOOR)
            H = float(finite_jacobian_meas(meas_gpr, x_pred))
            S = H * P_pred * H + r
            K = (P_pred * H / S) if S > 0 else 0.0
            x_upd = x_pred + K * (float(z) - h_pred)
            P_upd = (1.0 - K * H) * P_pred
            mode = 'measurement_update'
        else:
            h_pred = np.nan
            r = np.nan
            H = np.nan
            K = 0.0
            x_upd = x_pred
            P_upd = P_pred
            mode = 'prediction_only'

        x_upd = float(np.clip(x_upd, CLIP_SOH[0], CLIP_SOH[1]))
        P_upd = float(max(P_upd, 1e-12))

        t1 = time.perf_counter()
        update_ms = (t1 - t0) * 1000.0
        update_times_ms.append(update_ms)

        rows.append({
            'cell': row['cell'],
            'cycle_index': cycle,
            'y_true': y_true,
            'soh_pred': x_upd,
            'abs_error': abs(y_true - x_upd),
            'pmax_norm': z,
            'availability_prob': availability_prob,
            'measurement_available': measurement_available,
            'mode': mode,
            'x_pred_prior': x_pred,
            'P_pred': P_pred,
            'P_updated': P_upd,
            'q_from_gpr': q,
            'r_from_gpr': r,
            'F': F,
            'H': H,
            'K': K,
            'h_pred': h_pred,
            'update_time_ms': update_ms,
        })

        x, P = x_upd, P_upd
        prev_cycle = cycle

    pred_df = pd.DataFrame(rows)
    m = regression_metrics(pred_df['y_true'], pred_df['soh_pred'])
    m.update({
        'n_updates': int((pred_df['mode'] == 'measurement_update').sum()),
        'n_prediction_only': int((pred_df['mode'] == 'prediction_only').sum()),
        'mean_update_time_ms': float(np.mean(update_times_ms)) if update_times_ms else np.nan,
        'median_update_time_ms': float(np.median(update_times_ms)) if update_times_ms else np.nan,
        'p95_update_time_ms': float(np.percentile(update_times_ms, 95)) if update_times_ms else np.nan,
    })
    return pred_df, m


def evaluate_dataset_kadem_style(dataset_name, cfg):
    print(f'================ {dataset_name}: Kadem-style GPR-EKF BMS baseline ================')
    df = load_feature_table(dataset_name, cfg)
    print('Loaded rows:', len(df), 'cells:', sorted(df['cell'].unique()))
    print('Using single HI column:', 'pmax_norm_' + cfg['pmax_window'])

    all_pred_rows = []
    fold_rows = []
    model_rows = []

    cells = sorted(df['cell'].unique())
    for test_cell in tqdm(cells, desc=f'{dataset_name} LOCO'):
        train_df = df[df['cell'] != test_cell].copy()
        test_df = df[df['cell'] == test_cell].copy()

        tfit0 = time.perf_counter()
        pred_gpr, meas_gpr = fit_kadem_gpr_ekf_models(train_df, random_state=RANDOM_SEED)
        fit_ms = (time.perf_counter() - tfit0) * 1000.0
        model_size = pickle_size_kb({'pred_gpr': pred_gpr, 'meas_gpr': meas_gpr})
        model_rows.append({
            'dataset': dataset_name,
            'test_cell': test_cell,
            'train_rows': len(train_df),
            'test_rows': len(test_df),
            'fit_time_ms': fit_ms,
            'model_size_kb': model_size,
        })

        for p in AVAILABILITY_PROBS:
            seeds = [RANDOM_SEED] if p >= 0.999 else [RANDOM_SEED + i for i in range(N_SEEDS_FOR_DROPOUT)]
            for seed in seeds:
                pred_df, m = run_kadem_gpr_ekf_on_cell(test_df, pred_gpr, meas_gpr, availability_prob=p, seed=seed)
                pred_df['dataset'] = dataset_name
                pred_df['test_cell'] = test_cell
                pred_df['seed'] = seed
                pred_df['paper_pmax_window'] = cfg['pmax_window']
                all_pred_rows.append(pred_df)

                row = {
                    'dataset': dataset_name,
                    'test_cell': test_cell,
                    'availability_prob': p,
                    'seed': seed,
                    'paper_pmax_window': cfg['pmax_window'],
                    'paper_gpr_ekf_rmse': cfg['paper_gpr_ekf_rmse'],
                    'paper_gpr_ekf_mae': cfg['paper_gpr_ekf_mae'],
                }
                row.update(m)
                fold_rows.append(row)

    pred_all = pd.concat(all_pred_rows, ignore_index=True) if all_pred_rows else pd.DataFrame()
    fold_df = pd.DataFrame(fold_rows)
    model_df = pd.DataFrame(model_rows)

    # Aggregate by dataset and availability probability.
    # Metrics are first aggregated over cell/seed rows for traceability.
    summary = (
        fold_df
        .groupby(['dataset', 'availability_prob'], as_index=False)
        .agg(
            RMSE_mean=('RMSE', 'mean'), RMSE_std=('RMSE', 'std'),
            MAE_mean=('MAE', 'mean'), MAE_std=('MAE', 'std'),
            R2_mean=('R2', 'mean'), R2_std=('R2', 'std'),
            n_updates_mean=('n_updates', 'mean'),
            n_prediction_only_mean=('n_prediction_only', 'mean'),
            mean_update_time_ms=('mean_update_time_ms', 'mean'),
            median_update_time_ms=('median_update_time_ms', 'mean'),
            p95_update_time_ms=('p95_update_time_ms', 'mean'),
            paper_gpr_ekf_rmse=('paper_gpr_ekf_rmse', 'first'),
            paper_gpr_ekf_mae=('paper_gpr_ekf_mae', 'first'),
        )
    )
    summary['rmse_vs_paper_percent'] = 100.0 * (summary['paper_gpr_ekf_rmse'] - summary['RMSE_mean']) / summary['paper_gpr_ekf_rmse']
    summary['mae_vs_paper_percent'] = 100.0 * (summary['paper_gpr_ekf_mae'] - summary['MAE_mean']) / summary['paper_gpr_ekf_mae']

    size_summary = (
        model_df.groupby('dataset', as_index=False)
        .agg(model_size_kb_mean=('model_size_kb', 'mean'), fit_time_ms_mean=('fit_time_ms', 'mean'))
    )
    summary = summary.merge(size_summary, on='dataset', how='left')

    out_dir = os.path.join(RESULT_ROOT, dataset_name.replace(' ', '_'))
    os.makedirs(out_dir, exist_ok=True)
    pred_all.to_csv(os.path.join(out_dir, 'kadem_style_gpr_ekf_predictions.csv'), index=False)
    fold_df.to_csv(os.path.join(out_dir, 'kadem_style_gpr_ekf_fold_results.csv'), index=False)
    model_df.to_csv(os.path.join(out_dir, 'kadem_style_gpr_ekf_model_sizes.csv'), index=False)
    summary.to_csv(os.path.join(out_dir, 'kadem_style_gpr_ekf_summary.csv'), index=False)
    with pd.ExcelWriter(os.path.join(out_dir, 'kadem_style_gpr_ekf_results.xlsx')) as writer:
        summary.to_excel(writer, sheet_name='summary', index=False)
        fold_df.to_excel(writer, sheet_name='fold_results', index=False)
        model_df.to_excel(writer, sheet_name='model_sizes', index=False)
        pred_all.head(100000).to_excel(writer, sheet_name='predictions_head', index=False)

    print('Summary:')
    display(summary)
    return pred_all, fold_df, model_df, summary


In [ ]:
# ============================================================
# 5) RUN ALL THREE DATASETS
# ============================================================
all_summaries = []
all_folds = []
all_models = []

for dataset_name, cfg in DATASET_CONFIGS.items():
    pred_all, fold_df, model_df, summary = evaluate_dataset_kadem_style(dataset_name, cfg)
    all_summaries.append(summary)
    all_folds.append(fold_df)
    all_models.append(model_df)

combined_summary = pd.concat(all_summaries, ignore_index=True)
combined_folds = pd.concat(all_folds, ignore_index=True)
combined_models = pd.concat(all_models, ignore_index=True)

combined_summary_path = os.path.join(RESULT_ROOT, 'combined_kadem_style_gpr_ekf_summary.csv')
combined_excel_path = os.path.join(RESULT_ROOT, 'combined_kadem_style_gpr_ekf_results.xlsx')
combined_summary.to_csv(combined_summary_path, index=False)
with pd.ExcelWriter(combined_excel_path) as writer:
    combined_summary.to_excel(writer, sheet_name='summary', index=False)
    combined_folds.to_excel(writer, sheet_name='fold_results', index=False)
    combined_models.to_excel(writer, sheet_name='model_sizes', index=False)

print('================ COMBINED SUMMARY ================')
display(combined_summary)
print('Saved:', combined_summary_path)
print('Saved:', combined_excel_path)


In [ ]:
# ============================================================
# 6) OPTIONAL: COMPARISON-READY TABLE FOR THE MANUSCRIPT
# ============================================================
comparison = combined_summary.copy()
comparison['Protocol'] = comparison['availability_prob'].map(lambda p: 'Full HI availability' if p == 1.0 else f'Intermittent HI availability p={p:.1f}')
comparison = comparison[[
    'dataset', 'Protocol',
    'RMSE_mean', 'RMSE_std', 'MAE_mean', 'MAE_std',
    'paper_gpr_ekf_rmse', 'paper_gpr_ekf_mae',
    'rmse_vs_paper_percent',
    'mean_update_time_ms', 'median_update_time_ms', 'p95_update_time_ms',
    'model_size_kb_mean', 'n_updates_mean', 'n_prediction_only_mean'
]].copy()

display(comparison)
comparison.to_csv(os.path.join(RESULT_ROOT, 'manuscript_ready_kadem_style_comparison.csv'), index=False)


## Notes for manuscript interpretation

- This notebook is a **controlled Python/Colab reimplementation** of the Kadem-style single-HI GPR-EKF baseline, not the original MATLAB code.
- Runtime values are measured in the current Colab/Python environment. They should be compared with the Multi-HI/MAW-GME BMS notebooks under the same runtime, not directly with the MATLAB CPU runtime reported in the paper.
- The `availability_prob=0.8` setting is included to emulate the intermittent HI availability test. The main robustness analysis of the Multi-HI manuscript should still emphasize physically interpretable structured partial-window scenarios.
